# 04 — Train one classifier per extracted feature set

This notebook trains a separate classifier for each extracted feature folder.

It does **not** concatenate features. Each model is evaluated independently:

```text
resnet50 features                  -> classifier -> test metrics
efficientnet_b0 features           -> classifier -> test metrics
densenet121 features               -> classifier -> test metrics
mobilenet_v3_large features        -> classifier -> test metrics
vit_b_16 features                  -> classifier -> test metrics
cytoimagenet_efficientnetb0 features -> classifier -> test metrics
```

Outputs are saved under:

```text
<PROJECT_ROOT>/Output/single_feature_classification_results/<feature_model_name>/
    best_classifier.pt
    scaler.pkl
    test_predictions_idx.npy
    test_predictions_names.npy
    test_true_idx.npy
    test_true_names.npy
    test_classifier_embeddings.npy
    confusion_matrix.npy
    classification_report.txt
    training_history.csv

<PROJECT_ROOT>/Output/single_feature_classification_results/summary.csv
```

In [ ]:
from pathlib import Path
import copy
import pickle
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

import matplotlib.pyplot as plt

# Change this path if your shared folder is different.
PROJECT_ROOT = Path(r"M:\Shared247\jratcliff")
OUTPUT_DIR = PROJECT_ROOT / "Output"

SPLIT_DIR = OUTPUT_DIR / "splits"
FEATURE_DIR = OUTPUT_DIR / "features"
RESULTS_DIR = OUTPUT_DIR / "single_feature_classification_results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 10
RANDOM_SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Device:", device)
print("Feature folder:", FEATURE_DIR)
print("Results folder:", RESULTS_DIR)


In [ ]:
class_names = np.load(SPLIT_DIR / "class_names.npy", allow_pickle=True)

y_train = np.load(SPLIT_DIR / "y_train_idx.npy")
y_val = np.load(SPLIT_DIR / "y_val_idx.npy")
y_test = np.load(SPLIT_DIR / "y_test_idx.npy")

y_train_names = np.load(SPLIT_DIR / "y_train_names.npy", allow_pickle=True)
y_val_names = np.load(SPLIT_DIR / "y_val_names.npy", allow_pickle=True)
y_test_names = np.load(SPLIT_DIR / "y_test_names.npy", allow_pickle=True)

num_classes = len(class_names)

print("Number of classes:", num_classes)
print("Train labels:", y_train.shape)
print("Validation labels:", y_val.shape)
print("Test labels:", y_test.shape)


In [ ]:
feature_folders = []

for folder in sorted(FEATURE_DIR.iterdir()):
    if not folder.is_dir():
        continue

    required_files = [
        folder / "train_features.npy",
        folder / "val_features.npy",
        folder / "test_features.npy",
    ]

    if all(path.exists() for path in required_files):
        feature_folders.append(folder)

print("Detected feature folders:")
for folder in feature_folders:
    print("-", folder.name)

if not feature_folders:
    raise FileNotFoundError(
        f"No valid feature folders found in {FEATURE_DIR}. "
        "Run 03_extract_pretrained_features.ipynb first."
    )


In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, X_features, y_idx):
        self.X = torch.tensor(X_features, dtype=torch.float32)
        self.y = torch.tensor(y_idx, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class SingleFeatureClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.feature_layers = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        embedding = self.feature_layers(x)
        logits = self.classifier(embedding)
        return logits

    def extract_embedding(self, x):
        return self.feature_layers(x)


In [ ]:
def make_loaders(X_train, X_val, X_test):
    train_dataset = FeatureDataset(X_train, y_train)
    val_dataset = FeatureDataset(X_val, y_val)
    test_dataset = FeatureDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader, test_loader


def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for batch_features, batch_labels in loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(batch_features)
            loss = criterion(logits, batch_labels)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * batch_features.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == batch_labels).sum().item()
            total += batch_labels.size(0)

    return total_loss / total, correct / total


def evaluate_with_predictions(model, loader):
    model.eval()

    all_embeddings = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_features, batch_labels in loader:
            batch_features = batch_features.to(device)

            embeddings = model.extract_embedding(batch_features)
            logits = model(batch_features)
            preds = torch.argmax(logits, dim=1)

            all_embeddings.append(embeddings.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_labels.numpy())

    all_embeddings = np.concatenate(all_embeddings, axis=0)
    all_preds = np.asarray(all_preds, dtype=np.int64)
    all_labels = np.asarray(all_labels, dtype=np.int64)

    return all_embeddings, all_preds, all_labels


In [ ]:
def train_classifier_for_feature_folder(feature_folder):
    feature_name = feature_folder.name
    result_dir = RESULTS_DIR / feature_name
    result_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 100)
    print("Training classifier for feature:", feature_name)
    print("=" * 100)

    X_train_raw = np.load(feature_folder / "train_features.npy").astype(np.float32)
    X_val_raw = np.load(feature_folder / "val_features.npy").astype(np.float32)
    X_test_raw = np.load(feature_folder / "test_features.npy").astype(np.float32)

    print("Raw feature shapes:")
    print("  train:", X_train_raw.shape)
    print("  val:", X_val_raw.shape)
    print("  test:", X_test_raw.shape)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_val_scaled = scaler.transform(X_val_raw).astype(np.float32)
    X_test_scaled = scaler.transform(X_test_raw).astype(np.float32)

    with open(result_dir / "scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)

    train_loader, val_loader, test_loader = make_loaders(
        X_train_scaled,
        X_val_scaled,
        X_test_scaled,
    )

    input_dim = X_train_scaled.shape[1]

    model = SingleFeatureClassifier(
        input_dim=input_dim,
        num_classes=num_classes,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    history = []
    best_val_acc = -1.0
    best_state = None
    best_epoch = -1
    epochs_without_improvement = 0

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = run_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
        )

        val_loss, val_acc = run_one_epoch(
            model,
            val_loader,
            criterion,
            optimizer=None,
        )

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
        history.append(row)

        print(
            f"Epoch [{epoch + 1:02d}/{NUM_EPOCHS}] "
            f"Train Loss: {train_loss:.4f} "
            f"Train Acc: {train_acc:.4f} "
            f"Val Loss: {val_loss:.4f} "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}.")
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(result_dir / "training_history.csv", index=False)

    if best_state is not None:
        model.load_state_dict(best_state)

    test_embeddings, test_preds, test_labels = evaluate_with_predictions(
        model,
        test_loader,
    )

    test_acc = accuracy_score(test_labels, test_preds)
    macro_f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(test_labels, test_preds, average="weighted", zero_division=0)

    report_text = classification_report(
        test_labels,
        test_preds,
        labels=np.arange(num_classes),
        target_names=class_names,
        zero_division=0,
    )

    cm = confusion_matrix(
        test_labels,
        test_preds,
        labels=np.arange(num_classes),
    )

    pred_names = np.asarray([class_names[i] for i in test_preds])
    true_names = np.asarray([class_names[i] for i in test_labels])

    np.save(result_dir / "test_predictions_idx.npy", test_preds)
    np.save(result_dir / "test_predictions_names.npy", pred_names)
    np.save(result_dir / "test_true_idx.npy", test_labels)
    np.save(result_dir / "test_true_names.npy", true_names)
    np.save(result_dir / "test_classifier_embeddings.npy", test_embeddings)
    np.save(result_dir / "confusion_matrix.npy", cm)

    with open(result_dir / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report_text)

    torch.save(
        {
            "feature_name": feature_name,
            "input_dim": input_dim,
            "num_classes": num_classes,
            "class_names": class_names.tolist(),
            "best_epoch": best_epoch,
            "best_val_acc": best_val_acc,
            "test_acc": test_acc,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "model_state_dict": model.state_dict(),
        },
        result_dir / "best_classifier.pt",
    )

    print("\nTest results for:", feature_name)
    print("Best epoch:", best_epoch)
    print("Best validation accuracy:", best_val_acc)
    print("Test accuracy:", test_acc)
    print("Macro F1:", macro_f1)
    print("Weighted F1:", weighted_f1)
    print("\nClassification report:")
    print(report_text)

    summary_row = {
        "feature_name": feature_name,
        "input_dim": input_dim,
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "result_dir": str(result_dir),
    }

    return summary_row


In [ ]:
summary_rows = []

for feature_folder in feature_folders:
    summary_row = train_classifier_for_feature_folder(feature_folder)
    summary_rows.append(summary_row)

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("test_acc", ascending=False)

summary_path = RESULTS_DIR / "summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nSaved summary to:", summary_path)
display(summary_df)


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(summary_df["feature_name"], summary_df["test_acc"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Test accuracy")
plt.title("Single-feature classifier comparison")
plt.tight_layout()
plt.show()


In [ ]:
# Optional: display one confusion matrix with counts and row percentages.

from sklearn.metrics import ConfusionMatrixDisplay

best_feature_name = summary_df.iloc[0]["feature_name"]
best_result_dir = RESULTS_DIR / best_feature_name
cm = np.load(best_result_dir / "confusion_matrix.npy")

row_sums = cm.sum(axis=1, keepdims=True)
cm_percent = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

plt.figure(figsize=(10, 10))
plt.imshow(cm)
plt.title(f"Confusion matrix: {best_feature_name}\nTest accuracy = {summary_df.iloc[0]['test_acc']:.4f}")
plt.xlabel("Predicted condition")
plt.ylabel("True condition")
plt.xticks(range(num_classes), class_names, rotation=90)
plt.yticks(range(num_classes), class_names)
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        text = f"{cm[i, j]}\n{cm_percent[i, j] * 100:.1f}%"
        plt.text(
            j,
            i,
            text,
            ha="center",
            va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black",
            fontsize=8,
        )

plt.tight_layout()
plt.show()
